In [2]:
import pandas as pd
import numpy as np
import json
import ast
import os
import gc
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = r"D:\data_driven_marketing\train_v2.csv"
OUTPUT_FILE = "funnel_session_dataset.csv"
CHUNK_SIZE = 100_000

os.makedirs("funnel_chunks", exist_ok=True)

In [3]:
# ============================================================
# SAFE PARSERS
# ============================================================

def safe_parse_hits(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        return ast.literal_eval(x)
    except Exception:
        try:
            return json.loads(x)
        except Exception:
            return []


def safe_parse_json(x):
    if pd.isna(x):
        return {}

    if isinstance(x, dict):
        return x

    try:
        return json.loads(x)
    except Exception:
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}

In [4]:
def extract_funnel_from_hits(hits):
    page_paths = []
    action_types = []
    hit_types = []
    product_names = []
    product_categories = []

    for h in hits:
        if not isinstance(h, dict):
            continue

        # ---- hit type
        ht = h.get("type")
        if pd.notna(ht):
            hit_types.append(ht)

        # ---- page path
        page = h.get("page", {})
        if isinstance(page, dict):
            pp = page.get("pagePath")
            if pd.notna(pp):
                page_paths.append(pp)

        # ---- eCommerce action_type (INT, giữ thứ tự)
        ecom = h.get("eCommerceAction", {})
        if isinstance(ecom, dict):
            at = ecom.get("action_type", 0)
            try:
                at = int(at)
            except:
                at = 0
            action_types.append(at)

        # ---- products
        products = h.get("product", [])
        if isinstance(products, list):
            for p in products:
                if isinstance(p, dict):
                    pn = p.get("v2ProductName")
                    pc = p.get("v2ProductCategory")

                    if pd.notna(pn):
                        product_names.append(pn)
                    if pd.notna(pc):
                        product_categories.append(pc)

    # ============================================================
    # FEATURES (ĐÚNG mapping Google)
    # ============================================================

    has_click = int(1 in action_types)              # product list click
    has_product_view = int(2 in action_types)       # detail view
    has_add_to_cart = int(3 in action_types)        # add
    has_remove_from_cart = int(4 in action_types)   # remove
    has_checkout = int(5 in action_types)           # checkout
    has_purchase_action = int(6 in action_types)    # purchase
    has_refund = int(7 in action_types)             # refund
    has_checkout_option = int(8 in action_types)    # checkout option

    # ---- page-based fallback (rất hữu ích)
    has_cart_page = int(any(
        "basket" in str(p).lower() or "cart" in str(p).lower()
        for p in page_paths
    ))

    has_checkout_page = int(any(
        "checkout" in str(p).lower()
        for p in page_paths
    ))

    # ============================================================
    # OUTPUT
    # ============================================================

    return pd.Series({
        # ---- sequences (GIỮ LIST, KHÔNG JOIN STRING)
        "page_path_sequence": page_paths,
        "action_type_sequence": action_types,
        "hit_type_sequence": hit_types,
        "product_name_sequence": product_names,
        "product_category_sequence": product_categories,

        # ---- counts
        "n_hits_from_sequence": len(hits),
        "n_page_paths": len(page_paths),
        "n_unique_page_paths": len(set(page_paths)),
        "n_product_names": len(product_names),
        "n_unique_product_names": len(set(product_names)),

        # ---- funnel flags (ĐÚNG mapping)
        "has_click": has_click,
        "has_product_view": has_product_view,
        "has_add_to_cart": has_add_to_cart,
        "has_remove_from_cart": has_remove_from_cart,
        "has_checkout": has_checkout,
        "has_checkout_option": has_checkout_option,
        "has_purchase_action": has_purchase_action,
        "has_refund": has_refund,

        # ---- page-based signals
        "has_cart_page": has_cart_page,
        "has_checkout_page": has_checkout_page,

        # ---- abandonment
        "cart_abandonment": int(has_add_to_cart == 1 and has_purchase_action == 0),
        "checkout_abandonment": int(has_checkout == 1 and has_purchase_action == 0),
    })

In [5]:
# ============================================================
# PROCESS ONE CHUNK
# ============================================================

def process_funnel_chunk(chunk):
    chunk = chunk.copy()

    # ---- Basic time columns
    chunk["event_date"] = pd.to_datetime(
        chunk["date"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

    chunk["visitStartTime_datetime"] = pd.to_datetime(
        chunk["visitStartTime"],
        unit="s",
        errors="coerce"
    )

    # ---- Parse totals for revenue
    totals = chunk["totals"].apply(safe_parse_json)

    chunk["revenue_raw"] = totals.apply(
        lambda x: float(x.get("transactionRevenue", 0) or 0)
    )

    chunk["transactionRevenue_dollar"] = chunk["revenue_raw"] / 1_000_000
    chunk["target_log_revenue"] = np.log1p(chunk["revenue_raw"])

    # ---- Parse hits sequence
    parsed_hits = chunk["hits"].apply(safe_parse_hits)

    funnel_features = parsed_hits.apply(extract_funnel_from_hits)

    # ---- Keep columns useful for funnel analysis
    keep_cols = [
        "fullVisitorId",
        "visitId",
        "visitNumber",
        "visitStartTime",
        "visitStartTime_datetime",
        "event_date",
        "channelGrouping",
        "revenue_raw",
        "transactionRevenue_dollar",
        "target_log_revenue",
    ]

    keep_cols = [c for c in keep_cols if c in chunk.columns]

    out = pd.concat(
        [
            chunk[keep_cols].reset_index(drop=True),
            funnel_features.reset_index(drop=True)
        ],
        axis=1
    )

    return out

In [ ]:
# ============================================================
# RUN CHUNK PIPELINE
# ============================================================

first = True

reader = pd.read_csv(
    INPUT_FILE,
    dtype={"fullVisitorId": str},
    usecols=[
        "fullVisitorId",
        "visitId",
        "visitNumber",
        "visitStartTime",
        "date",
        "channelGrouping",
        "totals",
        "hits",
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i}, raw shape = {chunk.shape}")

    funnel_chunk = process_funnel_chunk(chunk)

    chunk_path = f"funnel_chunks/funnel_{i}.pkl"
    funnel_chunk.to_pickle(chunk_path)

    funnel_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first else "a",
        header=first,
        index=False
    )

    print(f"Saved chunk {i}: {funnel_chunk.shape}")

    first = False

    del chunk
    del funnel_chunk
    gc.collect()

print("Done!")
print("Saved final CSV:", OUTPUT_FILE)

Processing chunk 0, raw shape = (100000, 8)
Saved chunk 0: (100000, 32)
Processing chunk 1, raw shape = (100000, 8)
Saved chunk 1: (100000, 32)
Processing chunk 2, raw shape = (100000, 8)
Saved chunk 2: (100000, 32)
Processing chunk 3, raw shape = (100000, 8)
Saved chunk 3: (100000, 32)
Processing chunk 4, raw shape = (100000, 8)
Saved chunk 4: (100000, 32)
Processing chunk 5, raw shape = (100000, 8)
Saved chunk 5: (100000, 32)
Processing chunk 6, raw shape = (100000, 8)
Saved chunk 6: (100000, 32)
Processing chunk 7, raw shape = (100000, 8)
Saved chunk 7: (100000, 32)
Processing chunk 8, raw shape = (100000, 8)
Saved chunk 8: (100000, 32)
Processing chunk 9, raw shape = (100000, 8)


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001602E4ED4E0>>
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Roaming\Python\Python310\site-packages\ipykernel\ipkernel.py", line 788, in _clean_thread_parent_frames
    if phase != "start":
KeyboardInterrupt: 


In [6]:
START_CHUNK = 9  # <-- chạy tiếp từ đây

reader = pd.read_csv(
    INPUT_FILE,
    dtype={"fullVisitorId": str},
    usecols=[
        "fullVisitorId",
        "visitId",
        "visitNumber",
        "visitStartTime",
        "date",
        "channelGrouping",
        "totals",
        "hits",
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for i, chunk in enumerate(reader):

    # 🔥 Skip các chunk đã chạy
    if i < START_CHUNK:
        continue

    print(f"Processing chunk {i}, raw shape = {chunk.shape}")

    funnel_chunk = process_funnel_chunk(chunk)

    # Save pickle (an toàn)
    chunk_path = f"funnel_chunks/funnel_{i}.pkl"
    funnel_chunk.to_pickle(chunk_path)

    # 🔥 Append vào CSV, KHÔNG header, KHÔNG overwrite
    funnel_chunk.to_csv(
        OUTPUT_FILE,
        mode="a",        # append
        header=False,    # không ghi lại header
        index=False
    )

    print(f"Saved chunk {i}: {funnel_chunk.shape}")

    del chunk
    del funnel_chunk
    gc.collect()

print("Done!")

Processing chunk 9, raw shape = (100000, 8)
Saved chunk 9: (100000, 32)
Processing chunk 10, raw shape = (100000, 8)
Saved chunk 10: (100000, 32)
Processing chunk 11, raw shape = (100000, 8)
Saved chunk 11: (100000, 32)
Processing chunk 12, raw shape = (100000, 8)
Saved chunk 12: (100000, 32)
Processing chunk 13, raw shape = (100000, 8)
Saved chunk 13: (100000, 32)
Processing chunk 14, raw shape = (100000, 8)
Saved chunk 14: (100000, 32)
Processing chunk 15, raw shape = (100000, 8)
Saved chunk 15: (100000, 32)
Processing chunk 16, raw shape = (100000, 8)
Saved chunk 16: (100000, 32)
Processing chunk 17, raw shape = (8337, 8)
Saved chunk 17: (8337, 32)
Done!


In [11]:
df_work = pd.read_csv(r"D:\data_driven_marketing\Notebooks\funnel_session_dataset.csv")
df_work.head()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_26524\3739063136.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_work = pd.read_csv(r"D:\data_driven_marketing\Notebooks\funnel_session_dataset.csv")


,fullVisitorId,visitId,visitNumber,visitStartTime,visitStartTime_datetime,event_date,channelGrouping,revenue_raw,transactionRevenue_dollar,target_log_revenue,...,has_add_to_cart,has_remove_from_cart,has_checkout,has_checkout_option,has_purchase_action,has_refund,has_cart_page,has_checkout_page,cart_abandonment,checkout_abandonment
0,3162355547410993243,1508198450,1,1508198450,2017-10-17 00:00:50,2017-10-16,Organic Search,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,8934116514970143966,1508176307,6,1508176307,2017-10-16 17:51:47,2017-10-16,Referral,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,7992466427990357681,1508201613,1,1508201613,2017-10-17 00:53:33,2017-10-16,Direct,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,9075655783635761930,1508169851,1,1508169851,2017-10-16 16:04:11,2017-10-16,Organic Search,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,6960673291025684308,1508190552,1,1508190552,2017-10-16 21:49:12,2017-10-16,Organic Search,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
df_work["fullVisitorId"].value_counts()

fullVisitorId
1957458976293878100    337
7282998257608986241    257
3884810646891698298    222
824839726118485274     218
7477638593794484792    170
                      ... 
6095547410502786759      1
5139693001636159357      1
1265010355341538509      1
827723778572114902       1
8020876680451290044      1
Name: count, Length: 1371486, dtype: int64

In [24]:
df_work.columns

Index(['fullVisitorId', 'visitId', 'visitNumber', 'visitStartTime',
       'visitStartTime_datetime', 'event_date', 'channelGrouping',
       'revenue_raw', 'transactionRevenue_dollar', 'target_log_revenue',
       'page_path_sequence', 'action_type_sequence', 'hit_type_sequence',
       'product_name_sequence', 'product_category_sequence',
       'n_hits_from_sequence', 'n_page_paths', 'n_unique_page_paths',
       'n_product_names', 'n_unique_product_names', 'has_click',
       'has_product_view', 'has_add_to_cart', 'has_remove_from_cart',
       'has_checkout', 'has_checkout_option', 'has_purchase_action',
       'has_refund', 'has_cart_page', 'has_checkout_page', 'cart_abandonment',
       'checkout_abandonment'],
      dtype='object')